# Naïve Bayes à la main — détecteur de spam

On code **tout nous-mêmes**, sans bibliothèque magique, pour voir chaque rouage :

1. **Entraînement** = compter les mots dans un corpus déjà étiqueté → un *tableau* de probabilités. C'est ça, « le modèle ».
2. **Prédiction** = pour un nouvel email, **additionner les `log P(mot | classe)`** de chaque mot, et choisir la classe au plus gros total.

Tout repose sur la propriété qu'on a creusée : les mots indépendants → les probas se **multiplient**, et le `log` transforme ce produit en une **somme** de scores.


## Le corpus d'entraînement (déjà étiqueté)

La matière première : des emails dont on connaît **déjà** la classe (`spam` ou `ham` = normal). C'est ce qu'on appelle l'apprentissage *supervisé*.


In [49]:
import math
from collections import Counter

corpus = [
    ("Gagnez de l'argent facile, cliquez ici", "spam"),
    ("Argent gratuit, offre exclusive, cliquez maintenant", "spam"),
    ("Felicitations vous avez gagne un prix, reclamez votre argent", "spam"),
    ("Offre gratuite, argent rapide, cliquez ici maintenant", "spam"),
    ("Promotion exclusive, gagnez de l'argent gratuit", "spam"),
    ("Salut, on se voit demain pour le dejeuner", "ham"),
    ("Peux-tu m'envoyer le rapport du projet", "ham"),
    ("La reunion est deplacee a quinze heures demain", "ham"),
    ("Merci pour ton aide sur le document hier", "ham"),
    ("On dejeune ensemble demain midi au resto", "ham"),
]

classes = ["spam", "ham"]
for texte, label in corpus:
    print(f"[{label:4}] {texte}")


[spam] Gagnez de l'argent facile, cliquez ici
[spam] Argent gratuit, offre exclusive, cliquez maintenant
[spam] Felicitations vous avez gagne un prix, reclamez votre argent
[spam] Offre gratuite, argent rapide, cliquez ici maintenant
[spam] Promotion exclusive, gagnez de l'argent gratuit
[ham ] Salut, on se voit demain pour le dejeuner
[ham ] Peux-tu m'envoyer le rapport du projet
[ham ] La reunion est deplacee a quinze heures demain
[ham ] Merci pour ton aide sur le document hier
[ham ] On dejeune ensemble demain midi au resto


## Découper en mots (tokenisation)

Version simple : tout en minuscules, on remplace l'apostrophe et la ponctuation par des espaces, puis on coupe sur les espaces. (Naïf, mais suffisant pour comprendre.)


In [50]:
def tokeniser(texte):
    texte = texte.lower().replace("'", " ")
    for p in ",.;:!?":
        texte = texte.replace(p, " ")
    return texte.split()

print(tokeniser("Gagnez de l'argent gratuit, maintenant !"))


['gagnez', 'de', 'l', 'argent', 'gratuit', 'maintenant']


## Phase 1 — l'entraînement = **compter**

On compte, **séparément pour chaque classe**, combien de fois chaque mot apparaît. On en déduit aussi :
- le **vocabulaire** (tous les mots vus), de taille `V`,
- le **total de mots** par classe,
- les **priors** `P(spam)` et `P(ham)` = la proportion de chaque classe.


In [51]:
# Étape 1 — COMPTAGE : combien de fois chaque mot apparaît, séparément par classe

# Compteur vide pour l'instant -> compteur = {"spam": Counter(), "ham": Counter()}
compteur = {c: Counter() for c in classes}

# compteur rempli = un dictionnaire  { classe : Counter(mot -> nb d'occurrences) }
for texte, label in corpus:
    compteur[label].update(tokeniser(texte))

print("Structure de 'compteur' : un Counter (sac de mots) par classe\n")
for c in classes:
    print(f"[{c}]")
    print(f"   {compteur[c]}\n")

Structure de 'compteur' : un Counter (sac de mots) par classe

[spam]
   Counter({'argent': 5, 'cliquez': 3, 'gagnez': 2, 'de': 2, 'l': 2, 'ici': 2, 'gratuit': 2, 'offre': 2, 'exclusive': 2, 'maintenant': 2, 'facile': 1, 'felicitations': 1, 'vous': 1, 'avez': 1, 'gagne': 1, 'un': 1, 'prix': 1, 'reclamez': 1, 'votre': 1, 'gratuite': 1, 'rapide': 1, 'promotion': 1})

[ham]
   Counter({'demain': 3, 'le': 3, 'on': 2, 'pour': 2, 'salut': 1, 'se': 1, 'voit': 1, 'dejeuner': 1, 'peux-tu': 1, 'm': 1, 'envoyer': 1, 'rapport': 1, 'du': 1, 'projet': 1, 'la': 1, 'reunion': 1, 'est': 1, 'deplacee': 1, 'a': 1, 'quinze': 1, 'heures': 1, 'merci': 1, 'ton': 1, 'aide': 1, 'sur': 1, 'document': 1, 'hier': 1, 'dejeune': 1, 'ensemble': 1, 'midi': 1, 'au': 1, 'resto': 1})



In [52]:
# Étape 2 — VOCABULAIRE : tous les mots uniques vus, TOUTES classes confondues
# Sa taille V sert au lissage de Laplace (le "+V" au dénominateur, étape suivante)
vocabulaire = set()
for c in classes:
    vocabulaire |= set(compteur[c])   # union des mots vus dans chaque classe
V = len(vocabulaire)

print(f"V = {V} mots uniques (spam + ham confondus)\n")
print(f"Aperçu trié : {sorted(vocabulaire)}")

V = 54 mots uniques (spam + ham confondus)

Aperçu trié : ['a', 'aide', 'argent', 'au', 'avez', 'cliquez', 'de', 'dejeune', 'dejeuner', 'demain', 'deplacee', 'document', 'du', 'ensemble', 'envoyer', 'est', 'exclusive', 'facile', 'felicitations', 'gagne', 'gagnez', 'gratuit', 'gratuite', 'heures', 'hier', 'ici', 'l', 'la', 'le', 'm', 'maintenant', 'merci', 'midi', 'offre', 'on', 'peux-tu', 'pour', 'prix', 'projet', 'promotion', 'quinze', 'rapide', 'rapport', 'reclamez', 'resto', 'reunion', 'salut', 'se', 'sur', 'ton', 'un', 'voit', 'votre', 'vous']


In [53]:
# Étape 3 — TOTAL DE MOTS par classe : on additionne les occurrences de chaque Counter
# (c'est le DÉNOMINATEUR de P(mot | classe) -> on reste "dans le monde" de la classe)
total_mots = {c: sum(compteur[c].values()) for c in classes}

print("total_mots (nb total de mots de chaque classe) :")
for c in classes:
    print(f"   {c} : {total_mots[c]} mots")

total_mots (nb total de mots de chaque classe) :
   spam : 36 mots
   ham : 38 mots


In [54]:
# Étape 4 — PRIORS : la proportion de documents de chaque classe (avant de lire les mots)
n_docs = Counter(label for _, label in corpus)   # nb de documents par classe
total_docs = len(corpus)
prior = {c: n_docs[c] / total_docs for c in classes}

print(f"n_docs      = {dict(n_docs)}")
print(f"total_docs  = {total_docs}\n")
print("prior (proba qu'un email AU HASARD soit de cette classe) :")
for c in classes:
    print(f"   P({c}) = {n_docs[c]}/{total_docs} = {prior[c]}")

n_docs      = {'spam': 5, 'ham': 5}
total_docs  = 10

prior (proba qu'un email AU HASARD soit de cette classe) :
   P(spam) = 5/10 = 0.5
   P(ham) = 5/10 = 0.5


Regardons le **tableau de comptage** pour quelques mots parlants. Tu vois bien que « argent », « cliquez », « gratuit » vivent côté spam, tandis que « demain », « le », « pour » vivent côté ham.


In [55]:
mots_montres = ["argent", "cliquez", "gratuit", "offre", "demain", "le", "pour", "rapport"]
print(f"{'mot':<12}{'spam':>6}{'ham':>6}")
print("-" * 24)
for m in mots_montres:
    print(f"{m:<12}{compteur['spam'][m]:>6}{compteur['ham'][m]:>6}")


mot           spam   ham
------------------------
argent           5     0
cliquez          3     0
gratuit          2     0
offre            2     0
demain           0     3
le               0     3
pour             0     2
rapport          0     1


## Des comptages aux probabilités — et le piège du `log(0)`

`P(mot | classe)` = la fraction que représente ce mot parmi tous les mots de la classe :

$$P(\text{mot} \mid \text{classe}) = \frac{\text{nb d'occurrences du mot dans la classe}}{\text{nb total de mots de la classe}}$$

**Problème :** un mot jamais vu dans une classe donnerait `P = 0`, donc `log(0)` = $-\infty$, ce qui ferait exploser tout le score. 

**Solution (lissage de Laplace) :** on ajoute `+1` à chaque comptage (comme si on avait vu chaque mot du vocabulaire une fois de plus). Plus aucune proba n'est nulle :

$$P(\text{mot} \mid \text{classe}) = \frac{\text{occurrences} + 1}{\text{total mots de la classe} + V}$$


In [56]:
def proba_mot(mot, c):
    return (compteur[c][mot] + 1) / (total_mots[c] + V)

# quelques exemples
for m in ["argent", "demain", "licorne"]:   # "licorne" n'existe nulle part
    print(f"P({m!r:10} | spam) = {proba_mot(m,'spam'):.4f}   "
          f"P({m!r:10} | ham) = {proba_mot(m,'ham'):.4f}")


P('argent'   | spam) = 0.0667   P('argent'   | ham) = 0.0109
P('demain'   | spam) = 0.0111   P('demain'   | ham) = 0.0435
P('licorne'  | spam) = 0.0111   P('licorne'  | ham) = 0.0109


## Phase 2 — la prédiction = **additionner les log-probas**

Pour un nouvel email, le score d'une classe est :

$$\text{score}(c) = \log P(c) \;+\; \sum_{\text{mot}} \log P(\text{mot} \mid c)$$

Chaque mot **ajoute** sa contribution (`log P(mot | c)`). C'est l'**additivité** en action : le produit des probas est devenu une somme de scores grâce au `log`. La classe au plus gros total gagne.

La fonction ci-dessous affiche le **détail mot par mot**, pour qu'on voie chaque vote.


In [57]:
def classifier_detaille(texte):
    mots = tokeniser(texte)
    print(f"Email : {texte!r}\n")
    print(f"{'element':<14}{'log P(.|spam)':>15}{'log P(.|ham)':>15}   penche")
    print("-" * 60)

    score = {c: math.log(prior[c]) for c in classes}
    print(f"{'PRIOR':<14}{math.log(prior['spam']):>15.3f}{math.log(prior['ham']):>15.3f}")

    for mot in mots:
        ls = math.log(proba_mot(mot, "spam"))
        lh = math.log(proba_mot(mot, "ham"))
        score["spam"] += ls
        score["ham"] += lh
        penche = "spam" if ls > lh else ("ham" if lh > ls else "=")
        print(f"{mot:<14}{ls:>15.3f}{lh:>15.3f}   {penche}")

    print("-" * 60)
    print(f"{'TOTAL':<14}{score['spam']:>15.3f}{score['ham']:>15.3f}")
    verdict = max(score, key=score.get)
    print(f"\n=> VERDICT : {verdict.upper()}  "
          f"(score spam={score['spam']:.2f}, ham={score['ham']:.2f})")
    return verdict


## Le test sur plusieurs documents

On classe des emails **jamais vus à l'entraînement**. Regarde la colonne « penche » : chaque mot tire d'un côté, et le total tranche.


In [58]:
classifier_detaille("Gagnez de l'argent gratuit maintenant");

Email : "Gagnez de l'argent gratuit maintenant"

element         log P(.|spam)   log P(.|ham)   penche
------------------------------------------------------------
PRIOR                  -0.693         -0.693
gagnez                 -3.401         -4.522   spam
de                     -3.401         -4.522   spam
l                      -3.401         -4.522   spam
argent                 -2.708         -4.522   spam
gratuit                -3.401         -4.522   spam
maintenant             -3.401         -4.522   spam
------------------------------------------------------------
TOTAL                 -20.407        -27.824

=> VERDICT : SPAM  (score spam=-20.41, ham=-27.82)


In [59]:
classifier_detaille("Peux-tu m'envoyer le rapport demain");

Email : "Peux-tu m'envoyer le rapport demain"

element         log P(.|spam)   log P(.|ham)   penche
------------------------------------------------------------
PRIOR                  -0.693         -0.693
peux-tu                -4.500         -3.829   ham
m                      -4.500         -3.829   ham
envoyer                -4.500         -3.829   ham
le                     -4.500         -3.135   ham
rapport                -4.500         -3.829   ham
demain                 -4.500         -3.135   ham
------------------------------------------------------------
TOTAL                 -27.692        -22.279

=> VERDICT : HAM  (score spam=-27.69, ham=-22.28)


In [60]:
classifier_detaille("Offre exclusive cliquez ici");

Email : 'Offre exclusive cliquez ici'

element         log P(.|spam)   log P(.|ham)   penche
------------------------------------------------------------
PRIOR                  -0.693         -0.693
offre                  -3.401         -4.522   spam
exclusive              -3.401         -4.522   spam
cliquez                -3.114         -4.522   spam
ici                    -3.401         -4.522   spam
------------------------------------------------------------
TOTAL                 -14.010        -18.780

=> VERDICT : SPAM  (score spam=-14.01, ham=-18.78)


In [61]:
classifier_detaille("On se voit demain pour le dejeuner");

Email : 'On se voit demain pour le dejeuner'

element         log P(.|spam)   log P(.|ham)   penche
------------------------------------------------------------
PRIOR                  -0.693         -0.693
on                     -4.500         -3.423   ham
se                     -4.500         -3.829   ham
voit                   -4.500         -3.829   ham
demain                 -4.500         -3.135   ham
pour                   -4.500         -3.423   ham
le                     -4.500         -3.135   ham
dejeuner               -4.500         -3.829   ham
------------------------------------------------------------
TOTAL                 -32.192        -25.296

=> VERDICT : HAM  (score spam=-32.19, ham=-25.30)


## Ce qu'il faut remarquer

- **Le « modèle », c'est juste `compteur`** : un tableau de comptages. Rien de magique.
- Chaque mot **ajoute** son `log P(mot | classe)`. Les mots typés (« argent », « cliquez ») penchent fort d'un côté ; les mots neutres (« de », « le ») penchent à peine.
- Le lissage `+1` évite le `log(0)` quand un mot est inconnu d'une classe.

### À toi de jouer
- Ajoute des emails au `corpus` et **ré-exécute** : le tableau change → le modèle « apprend » autrement.
- Teste un email ambigu et regarde la colonne « penche » pour comprendre la décision.
- Compare `score spam` et `score ham` : plus l'écart est grand, plus le modèle est « sûr ».
